In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb


In [ ]:
# --- Bước 1: Đọc dữ liệu ---
data = pd.read_csv('data.csv', parse_dates=['date'], index_col='date')
data = data.sort_index()

# --- Bước 2: Tiền xử lý dữ liệu ---
# Sử dụng MinMaxScaler để chuẩn hóa cột 'value'
scaler = MinMaxScaler(feature_range=(0, 1))
data['value_scaled'] = scaler.fit_transform(data[['value']])

# Tạo các đặc trưng lag: ví dụ sử dụng 3 giá trị trước để dự báo giá trị hiện tại
def create_lag_features(df, lags=3):
    df_features = pd.DataFrame()
    for lag in range(1, lags + 1):
        df_features[f'lag_{lag}'] = df['value_scaled'].shift(lag)
    df_features['target'] = df['value_scaled']
    return df_features.dropna()

lags = 3
df_features = create_lag_features(data, lags)

# Tách đặc trưng (X) và nhãn (y)
X = df_features.drop('target', axis=1).values
y = df_features['target'].values

# --- Bước 3: Chia dữ liệu thành tập huấn luyện và kiểm tra (80% - 20%) ---
# Lưu ý: Không xáo trộn dữ liệu để giữ tính tuần tự của chuỗi thời gian
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# --- Bước 4: Xây dựng và huấn luyện mô hình XGBoost ---
model = xgb.XGBRegressor(
    n_estimators=100,   # số lượng cây
    max_depth=3,        # độ sâu của cây
    learning_rate=0.1,  # tốc độ học
    objective='reg:squarederror',  # hàm loss cho hồi quy
    random_state=42
)
model.fit(X_train, y_train)

# Dự báo trên tập kiểm tra
y_pred = model.predict(X_test)

# Tính toán MSE
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.4f}")

# --- Bước 5: Chuyển đổi dự báo về giá trị ban đầu ---
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_inv = scaler.inverse_transform(y_pred.reshape(-1, 1))

# Trực quan hóa kết quả
plt.figure(figsize=(8, 5))
plt.plot(y_test_inv, label='Giá trị thực', marker='o')
plt.plot(y_pred_inv, label='Dự báo', marker='x', linestyle='--')
plt.title('Dự báo chuỗi thời gian với XGBoost')
plt.xlabel('Mẫu')
plt.ylabel('Giá trị')
plt.legend()
plt.show()

4. Giải thích chi tiết
Tiền xử lý dữ liệu:

Dữ liệu được đọc từ file data.csv và sắp xếp theo thứ tự thời gian.

Sử dụng MinMaxScaler để chuẩn hóa cột value về khoảng [0, 1].

Hàm create_lag_features tạo ra các đặc trưng lag (ở đây dùng 3 lag) để làm đầu vào cho mô hình, với nhãn là giá trị hiện tại.

Chia dữ liệu:

Dữ liệu được chia thành tập huấn luyện và kiểm tra theo tỷ lệ 80-20 mà không xáo trộn, giúp giữ tính tuần tự của chuỗi thời gian.

Xây dựng mô hình XGBoost:

Sử dụng xgb.XGBRegressor với các tham số như số lượng cây, độ sâu của cây, tốc độ học,...

Huấn luyện mô hình trên tập huấn luyện, sau đó dự báo trên tập kiểm tra.

Đánh giá và trực quan hóa:

Tính toán MSE để đánh giá hiệu năng mô hình.

Chuyển đổi kết quả dự báo và giá trị thực từ dạng chuẩn hóa về giá trị ban đầu, rồi vẽ biểu đồ so sánh.

